In [31]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

Setup done.

# 🏗️ Алгебры и Коалгебры в Haskell

## Структура рекурсии через категориальные абстракции

---

### 📚 Содержание

1. **F-алгебры** — что такое алгебра над функтором
2. **Начальные алгебры и катаморфизмы** — сворачиваем рекурсивные структуры
3. **F-коалгебры** — двойственная картина
4. **Терминальные коалгебры и анаморфизмы** — разворачиваем бесконечные структуры
5. **Гиломорфизмы** — build + fold = hylo
6. **Параморфизмы** — свёртка с доступом к оригинальной структуре
7. **Апоморфизмы** — развёртка с возможностью "прыгнуть"
8. **Гистоморфизмы** — свёртка с историей (курсор)
9. **Футуморфизмы** — развёртка с предсказаниями
10. **Синтез: схема рекурсии** — унификация через Fix
11. **Категориальная перспектива** — диаграммы и законы

---

> **Главная идея:** Разделить *форму рекурсии* и *логику шага*.
> F-алгебра описывает «что делать на одном уровне», а схема рекурсии — «как обойти всю структуру».

> **📦 Зависимости**
> **Пакеты:** `base`, `containers`, `process`
> **Расширения:** `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


In [32]:
import Data.List (intercalate, sort, nub)
import System.IO
import GHC.IO.Handle
let svgStr = "<svg xmlns='http://www.w3.org/2000/svg' width='300' height='150'><circle cx='150' cy='75' r='60' fill='lightblue' stroke='navy' stroke-width='3'/><text x='150' y='85' text-anchor='middle' font-size='18' fill='navy'>Hask</text></svg>"
writeFile "/tmp/hask_circle.svg" svgStr
-- Вспомогательные функции для форматирования вывода
putHeader :: String -> IO ()
putHeader s = do
  let border = replicate (length s + 4) '='
  putStrLn border
  putStrLn $ "= " ++ s ++ " ="
  putStrLn border
  putStrLn ""

putSub :: String -> IO ()
putSub s = do
  putStrLn $ "--- " ++ s ++ " ---"



In [33]:
:! cp /tmp/hask_circle.svg . && ls hask_circle.svg

hask_circle.svg

### Категория Hask

![Категория Hask — объекты (типы), морфизмы (функции)](../diagrams/haskell/hask_circle.svg)

![Коммутативная диаграмма F-алгебры (морфизм алгебр)](../diagrams/algebras/falgebra_diagram.svg)

In [34]:
import System.Process
import System.IO

let svgCode = "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"300\" height=\"200\"><defs><marker id=\"arr\" markerWidth=\"10\" markerHeight=\"7\" refX=\"10\" refY=\"3.5\" orient=\"auto\"><polygon points=\"0 0, 10 3.5, 0 7\" fill=\"black\"/></marker></defs><text x=\"30\" y=\"40\" font-family=\"serif\" font-size=\"18\" font-style=\"italic\">FA</text><text x=\"240\" y=\"40\" font-family=\"serif\" font-size=\"18\" font-style=\"italic\">FB</text><text x=\"30\" y=\"175\" font-family=\"serif\" font-size=\"18\" font-style=\"italic\">A</text><text x=\"240\" y=\"175\" font-family=\"serif\" font-size=\"18\" font-style=\"italic\">B</text><line x1=\"65\" y1=\"32\" x2=\"230\" y2=\"32\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/><text x=\"140\" y=\"25\" font-family=\"serif\" font-size=\"14\">Ff</text><line x1=\"65\" y1=\"165\" x2=\"230\" y2=\"165\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/><text x=\"145\" y=\"158\" font-family=\"serif\" font-size=\"14\">f</text><line x1=\"38\" y1=\"50\" x2=\"38\" y2=\"155\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/><text x=\"45\" y=\"107\" font-family=\"serif\" font-size=\"14\">&#945;</text><line x1=\"248\" y1=\"50\" x2=\"248\" y2=\"155\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/><text x=\"255\" y=\"107\" font-family=\"serif\" font-size=\"14\">&#946;</text></svg>"

-- Запишем SVG в файл и отобразим через img tag
writeFile "/tmp/diagram.svg" svgCode
putStrLn "SVG saved to /tmp/diagram.svg"

SVG saved to /tmp/diagram.svg

In [35]:
:! cp /tmp/diagram.svg . && pwd && ls *.svg 2>/dev/null

/home/jovyan/src
adj_adjunction.svg
adj_triangle.svg
cm_comonad.svg
cm_comonad_pairs.svg
cm_final.svg
conc_landscape.svg
design_dark_template.svg
diagram.svg
dimap_square.svg
dist_landscape.svg
dist_table.svg
falgebra_diagram.svg
falgebra_universal.svg
fh_hierarchy.svg
ft_compare.svg
ft_foldable.svg
gpu_landscape.svg
hask_circle.svg
hylo_tree.svg
initial_alg.svg
kan_adjunction.svg
kan_codensity.svg
kan_density.svg
kan_examples.svg
kan_yoneda.svg
lan_diagram.svg
monad_choice.svg
monad_hierarchy.svg
mp_generics.svg
mp_th_stages.svg
mt_diagram.svg
mt_layers.svg
mt_order.svg
mt_stack.svg
op_optics.svg
prof_dimap.svg
ran_diagram.svg
recursion_schemes.svg
ta_product_coproduct.svg
ta_semiring.svg
ta_zipper.svg
yo_yoneda.svg

### Коммутативная диаграмма F-алгебры

![Коммутативная диаграмма F-алгебры](../diagrams/misc/diagram.svg)

In [36]:
-- Тип F-алгебры
type Algebra f a = f a -> a

-- ============================================================
-- Пример 1: Натуральные числа
-- ============================================================
data NatF r = ZeroF | SuccF r deriving (Show, Functor)

-- Несколько разных алгебр над одним и тем же функтором NatF:

-- Алгебра «сложение»: NatF Int -> Int
natToInt :: Algebra NatF Int
natToInt ZeroF     = 0
natToInt (SuccF n) = n + 1

-- Алгебра «создание строки»
natToStr :: Algebra NatF String
natToStr ZeroF     = "zero"
natToStr (SuccF s) = "succ(" ++ s ++ ")"

-- Алгебра «булево: чётное?»
natIsEven :: Algebra NatF Bool
natIsEven ZeroF      = True
natIsEven (SuccF b)  = not b

-- Проверим алгебры вручную (без рекурсии):
-- Представим число 3 = Succ(Succ(Succ(Zero)))
-- F-алгебра применяется изнутри-наружу
main1 :: IO ()
main1 = do
  putHeader "F-Алгебры: примеры"

  putSub "natToInt — вычислить значение"
  let step0 = natToInt ZeroF          -- = 0
      step1 = natToInt (SuccF step0)  -- = 1
      step2 = natToInt (SuccF step1)  -- = 2
      step3 = natToInt (SuccF step2)  -- = 3
  putStrLn $ "Вручную: " ++ show step3

  putSub "natToStr — показать структуру"
  let s0 = natToStr ZeroF
      s1 = natToStr (SuccF s0)
      s2 = natToStr (SuccF s1)
      s3 = natToStr (SuccF s2)
  putStrLn $ "Структура: " ++ s3

  putSub "natIsEven — чётность"
  let e0 = natIsEven ZeroF
      e1 = natIsEven (SuccF e0)
      e2 = natIsEven (SuccF e1)
      e3 = natIsEven (SuccF e2)
      e4 = natIsEven (SuccF e3)
  putStrLn $ "0 чётное: " ++ show e0
  putStrLn $ "1 чётное: " ++ show e1
  putStrLn $ "4 чётное: " ++ show e4

main1

= F-Алгебры: примеры =

--- natToInt — вычислить значение ---
Вручную: 3
--- natToStr — показать структуру ---
Структура: succ(succ(succ(zero)))
--- natIsEven — чётность ---
0 чётное: True
1 чётное: False
4 чётное: True

In [37]:
-- ============================================================
-- Пример 2: Списки
-- ============================================================
data ListF a r = NilF | ConsF a r deriving (Show, Functor)

-- Алгебра «сумма»
sumAlg :: Algebra (ListF Int) Int
sumAlg NilF        = 0
sumAlg (ConsF x r) = x + r

-- Алгебра «длина»
lenAlg :: Algebra (ListF a) Int
lenAlg NilF        = 0
lenAlg (ConsF _ r) = r + 1

-- Алгебра «максимум»
maxAlg :: Algebra (ListF Int) (Maybe Int)
maxAlg NilF        = Nothing
maxAlg (ConsF x r) = Just $ maybe x (max x) r

-- Алгебра «реверс через разность»
revAlg :: Algebra (ListF a) ([a] -> [a])
revAlg NilF        = id
revAlg (ConsF x f) = f . (x:)

-- Демонстрация для списка [3, 1, 4, 1, 5]
main2 :: IO ()
main2 = do
  putHeader "Алгебры над ListF"

  putSub "Сумма [3,1,4,1,5]"
  let s = sumAlg (ConsF 3 (sumAlg (ConsF 1 (sumAlg (ConsF 4 (sumAlg (ConsF 1 (sumAlg (ConsF 5 (sumAlg NilF))))))))))
  putStrLn $ "Сумма: " ++ show s

  putSub "Длина [3,1,4,1,5]"
  let applyList alg [] = alg NilF
      applyList alg (x:xs) = alg (ConsF x (applyList alg xs))
  putStrLn $ "Длина: " ++ show (applyList lenAlg [3,1,4,1,5 :: Int])
  putStrLn $ "Максимум: " ++ show (applyList maxAlg [3,1,4,1,5 :: Int])

  putSub "Реверс через алгебру функций"
  let rev xs = applyList revAlg xs []
  putStrLn $ "reverse [1..5] = " ++ show (rev [1..5 :: Int])

main2

= Алгебры над ListF =

--- Сумма [3,1,4,1,5] ---
Сумма: 14
--- Длина [3,1,4,1,5] ---
Длина: 5
Максимум: Just 5
--- Реверс через алгебру функций ---
reverse [1..5] = [5,4,3,2,1]

## 2️⃣ Начальная Алгебра и Катаморфизмы

### 📐 Теория

**Начальная F-алгебра** `(μF, in)` — это алгебра, из которой существует **единственный гомоморфизм** в любую другую F-алгебру.

В Haskell начальная алгебра моделируется типом `Fix F`:
```haskell
newtype Fix f = Fix { unFix :: f (Fix f) }
```

**Катаморфизм** (от греч. κατά — «вниз») — обобщение `foldr`:
```haskell
cata :: Functor f => Algebra f a -> Fix f -> a
cata alg = alg . fmap (cata alg) . unFix
```

### ⭐ Диаграмма (коммутативный квадрат)

![Диаграмма F-алгебры](../diagrams/algebras/falgebra_universal.svg)

### 💡 Смысл

- `unFix` — разворачиваем один слой
- `fmap (cata alg)` — рекурсивно применяем ко всем подструктурам
- `alg` — применяем алгебру к уже обработанным результатам

### 🔭 Теорема Ламбека

Начальная алгебра `in :: F(μF) → μF` является **изоморфизмом**.
То есть `μF ≅ F(μF)` — тип рекурсивно изоморфен своему «развёрнутому» виду.

### Теорема Ламбека и уникальность

**Коммутативный квадрат** (диаграмма универсального свойства):

![Universal&#39;noe svojstvo](../diagrams/algebras/falgebra_universal.svg)

Закон коммутативности: `h . in = alg . fmap h`

Иными словами: **катаморфизм — это единственная стрелка** из (μF, in)

In [38]:
import System.Process
import System.IO

-- Diagram: Initial algebra universality
-- muF ---[cata alg]---> A
--  |                    |
-- [F(cata alg)]        [alg]
--  |                    |
--  v                    v
-- F(muF) ---[in]--->   F A
-- (bottom row: F(muF) and F A connected by F(cata alg))
-- (left col: F(muF) -> muF via "in")
-- (right col: F A -> A via "alg")
-- (top: muF -> A via unique "cata alg")

let svgCode = "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"360\" height=\"220\">" ++
      "<defs><marker id=\"arr\" markerWidth=\"10\" markerHeight=\"7\" refX=\"10\" refY=\"3.5\" orient=\"auto\">" ++
      "<polygon points=\"0 0, 10 3.5, 0 7\" fill=\"black\"/></marker></defs>" ++
      -- boxes: muF (top-left), A (top-right), F(muF) (bottom-left), F(A) (bottom-right)
      "<rect x=\"30\" y=\"20\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#e8f4ff\" stroke=\"#4488cc\" stroke-width=\"1.5\"/>" ++
      "<text x=\"70\" y=\"43\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"16\" font-style=\"italic\">&#956;F</text>" ++
      "<rect x=\"250\" y=\"20\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#fff4e8\" stroke=\"#cc8844\" stroke-width=\"1.5\"/>" ++
      "<text x=\"290\" y=\"43\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"16\" font-style=\"italic\">A</text>" ++
      "<rect x=\"30\" y=\"164\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#e8f4ff\" stroke=\"#4488cc\" stroke-width=\"1.5\"/>" ++
      "<text x=\"70\" y=\"187\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"14\">F(&#956;F)</text>" ++
      "<rect x=\"250\" y=\"164\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#fff4e8\" stroke=\"#cc8844\" stroke-width=\"1.5\"/>" ++
      "<text x=\"290\" y=\"187\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"14\">F(A)</text>" ++
      -- top arrow: muF -> A (unique cata alg)
      "<line x1=\"112\" y1=\"38\" x2=\"248\" y2=\"38\" stroke=\"#dd2222\" stroke-width=\"2\" stroke-dasharray=\"6,3\" marker-end=\"url(#arr)\"/>" ++
      "<text x=\"180\" y=\"30\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\" fill=\"#dd2222\">cata alg</text>" ++
      "<text x=\"180\" y=\"52\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"10\" fill=\"#888\">(unique!)</text>" ++
      -- bottom arrow: F(muF) -> F(A)  (F(cata alg))
      "<line x1=\"112\" y1=\"182\" x2=\"248\" y2=\"182\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/>" ++
      "<text x=\"180\" y=\"173\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\">F(cata alg)</text>" ++
      -- left arrow: F(muF) -> muF  (in)
      "<line x1=\"70\" y1=\"162\" x2=\"70\" y2=\"58\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/>" ++
      "<text x=\"48\" y=\"112\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\">in</text>" ++
      -- right arrow: F(A) -> A  (alg)
      "<line x1=\"290\" y1=\"162\" x2=\"290\" y2=\"58\" stroke=\"black\" stroke-width=\"1.5\" marker-end=\"url(#arr)\"/>" ++
      "<text x=\"315\" y=\"112\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\">alg</text>" ++
      -- title
      "<text x=\"180\" y=\"215\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"11\" fill=\"#555\">Universality of initial algebra</text>" ++
      "</svg>"

writeFile "/tmp/initial_alg.svg" svgCode
putStrLn "Initial algebra diagram saved."

Initial algebra diagram saved.

In [39]:
:! cp /tmp/initial_alg.svg /home/jovyan/src/ && echo "copied initial_alg.svg"

copied initial_alg.svg

### Диаграмма: Универсальность начальной алгебры

![Начальная алгебра: универсальность катаморфизма](../diagrams/algebras/initial_alg.svg)

Диаграмма коммутирует: `alg . F(cata alg) = cata alg . in`  
Для **любой** F-алгебры `alg :: F A -> A` существует **единственная** стрелка `cata alg :: muF -> A`.

In [40]:
-- ============================================================
-- Fix — точка неподвижности функтора
-- ============================================================
newtype Fix f = Fix { unFix :: f (Fix f) }

-- Катаморфизм — обобщённый fold
cata :: Functor f => Algebra f a -> Fix f -> a
cata alg = alg . fmap (cata alg) . unFix

-- ============================================================
-- Натуральные числа через Fix
-- ============================================================
-- type Nat = Fix NatF
zero :: Fix NatF
zero = Fix ZeroF

succ' :: Fix NatF -> Fix NatF
succ' n = Fix (SuccF n)

-- Строим числа
one, two, three, four, five :: Fix NatF
one   = succ' zero
two   = succ' one
three = succ' two
four  = succ' three
five  = succ' four

-- Катаморфизмы для натуральных чисел
natVal :: Fix NatF -> Int
natVal = cata natToInt

natStr :: Fix NatF -> String
natStr = cata natToStr

natEven :: Fix NatF -> Bool
natEven = cata natIsEven

-- Алгебра сложения (через кодировку Черча)
addAlg :: Int -> Algebra NatF Int
addAlg m ZeroF     = m
addAlg _ (SuccF n) = n + 1

addNat :: Fix NatF -> Int -> Int
addNat n m = cata (addAlg m) n

-- Демонстрация
main3 :: IO ()
main3 = do
  putHeader "Катаморфизм для Nat"
  putStrLn $ "natVal five = " ++ show (natVal five)
  putStrLn $ "natStr three = " ++ natStr three
  putStrLn $ "natEven four = " ++ show (natEven four)
  putStrLn $ "2 + 3 = " ++ show (addNat two 3)
  putStrLn $ "5 + 5 = " ++ show (addNat five 5)

main3

= Катаморфизм для Nat =

natVal five = 5
natStr three = succ(succ(succ(zero)))
natEven four = True
2 + 3 = 5
5 + 5 = 10

In [41]:
-- =================================================================
-- Universalnoe svojstvo: cata alg -- edinstvennyj gomomorifzm
-- =================================================================

-- Zakon gomomorifzma F-algebr:
-- h . in_alg = alg . fmap h
-- Dlya cata: cata alg . Fix = alg . fmap (cata alg)

homLaw :: IO ()
homLaw = do
  putHeader "Zakon gomomorifzma F-algebr"
  let v0 = ZeroF :: NatF (Fix NatF)
  let v1 = SuccF zero :: NatF (Fix NatF)
  let v2 = SuccF one :: NatF (Fix NatF)
  -- Left: cata natToInt . Fix
  let lhs x = cata natToInt (Fix x)
  -- Right: natToInt . fmap (cata natToInt)
  let rhs x = natToInt (fmap (cata natToInt) x)
  putStrLn $ "lhs v0 = " ++ show (lhs v0)
  putStrLn $ "rhs v0 = " ++ show (rhs v0)
  putStrLn $ "lhs v1 = " ++ show (lhs v1)
  putStrLn $ "rhs v1 = " ++ show (rhs v1)
  putStrLn $ "lhs v2 = " ++ show (lhs v2)
  putStrLn $ "rhs v2 = " ++ show (rhs v2)
  putStrLn $ "Law holds: " ++ show (lhs v0 == rhs v0 && lhs v1 == rhs v1 && lhs v2 == rhs v2)

homLaw


= Zakon gomomorifzma F-algebr =

lhs v0 = 0
rhs v0 = 0
lhs v1 = 1
rhs v1 = 1
lhs v2 = 2
rhs v2 = 2
Law holds: True

In [42]:
-- ============================================================
-- Списки через Fix
-- ============================================================
-- type List a = Fix (ListF a)

nilL :: Fix (ListF a)
nilL = Fix NilF

consL :: a -> Fix (ListF a) -> Fix (ListF a)
consL x xs = Fix (ConsF x xs)

-- Строим список [1,2,3,4,5]
fromList :: [a] -> Fix (ListF a)
fromList []     = nilL
fromList (x:xs) = consL x (fromList xs)

toList :: Fix (ListF a) -> [a]
toList = cata alg
  where
    alg NilF        = []
    alg (ConsF x r) = x : r

-- Катаморфизмы для списков
sumList :: Fix (ListF Int) -> Int
sumList = cata sumAlg

lengthList :: Fix (ListF a) -> Int
lengthList = cata lenAlg

mapList :: (a -> b) -> Fix (ListF a) -> Fix (ListF b)
mapList f = cata alg
  where
    alg NilF        = nilL
    alg (ConsF x r) = consL (f x) r

filterList :: (a -> Bool) -> Fix (ListF a) -> Fix (ListF a)
filterList p = cata alg
  where
    alg NilF        = nilL
    alg (ConsF x r) = if p x then consL x r else r

-- Алгебра: сортировка вставками
insertSortAlg :: (Ord a) => Algebra (ListF a) [a]
insertSortAlg NilF        = []
insertSortAlg (ConsF x r) = insertSorted x r
  where
    insertSorted y []     = [y]
    insertSorted y (z:zs) = if y <= z then y : z : zs else z : insertSorted y zs

insertionSort :: Ord a => Fix (ListF a) -> [a]
insertionSort = cata insertSortAlg

-- Демонстрация
main4 :: IO ()
main4 = do
  putHeader "Катаморфизм для List"
  let lst = fromList [3,1,4,1,5,9,2,6 :: Int]
  putStrLn $ "toList: " ++ show (toList lst)
  putStrLn $ "sum: " ++ show (sumList lst)
  putStrLn $ "length: " ++ show (lengthList lst)
  putStrLn $ "map (*2): " ++ show (toList $ mapList (*2) lst)
  putStrLn $ "filter even: " ++ show (toList $ filterList even lst)
  putStrLn $ "insertionSort: " ++ show (insertionSort lst)

main4

Line 14: Use foldr
Found:
fromList [] = nilL
fromList (x : xs) = consL x (fromList xs)
Why not:
fromList xs = foldr consL nilL xs

= Катаморфизм для List =

toList: [3,1,4,1,5,9,2,6]
sum: 31
length: 8
map (*2): [6,2,8,2,10,18,4,12]
filter even: [4,2,6]
insertionSort: [1,1,2,3,4,5,6,9]

In [43]:
-- ============================================================
-- Деревья через Fix
-- ============================================================
data TreeF a r = LeafF | NodeF r a r deriving (Show, Functor)

-- type Tree a = Fix (TreeF a)
leaf :: Fix (TreeF a)
leaf = Fix LeafF

node :: Fix (TreeF a) -> a -> Fix (TreeF a) -> Fix (TreeF a)
node l x r = Fix (NodeF l x r)

-- Строим дерево поиска
fromSortedList :: [a] -> Fix (TreeF a)
fromSortedList [] = leaf
fromSortedList xs =
  let mid = length xs `div` 2
      (ls, m:rs) = splitAt mid xs
  in node (fromSortedList ls) m (fromSortedList rs)

-- Катаморфизмы для деревьев
treeSum :: Fix (TreeF Int) -> Int
treeSum = cata alg
  where
    alg LeafF        = 0
    alg (NodeF l x r) = l + x + r

treeDepth :: Fix (TreeF a) -> Int
treeDepth = cata alg
  where
    alg LeafF         = 0
    alg (NodeF l _ r) = 1 + max l r

treeInOrder :: Fix (TreeF a) -> [a]
treeInOrder = cata alg
  where
    alg LeafF         = []
    alg (NodeF l x r) = l ++ [x] ++ r

-- Красивый показ дерева
showTree :: Show a => Fix (TreeF a) -> String
showTree = cata alg
  where
    alg LeafF         = "."
    alg (NodeF l x r) = "(" ++ l ++ " " ++ show x ++ " " ++ r ++ ")"

main5 :: IO ()
main5 = do
  putHeader "Катаморфизм для Tree"
  let t = fromSortedList [1..7 :: Int]
  putStrLn $ "Структура: " ++ showTree t
  putStrLn $ "In-order:  " ++ show (treeInOrder t)
  putStrLn $ "Сумма:     " ++ show (treeSum t)
  putStrLn $ "Глубина:   " ++ show (treeDepth t)

main5

= Катаморфизм для Tree =

Структура: (((. 1 .) 2 (. 3 .)) 4 ((. 5 .) 6 (. 7 .)))
In-order:  [1,2,3,4,5,6,7]
Сумма:     28
Глубина:   3

## 3️⃣ F-Коалгебры

### 📐 Теория

**F-коалгебра** — это пара `(A, α)`, где:
- `F` — тот же функтор
- `A` — тип-носитель (seed / generator)
- `α :: A → F A` — функция-развёртка (одно направление стрелки перевёрнуто!)

F-алгебра и F-коалгебра — **точные двойники** (категориальная дуальность):

| F-Алгебра | F-Коалгебра |
|-----------|-------------|
| `F A → A` | `A → F A` |
| Начальная алгебра | Терминальная коалгебра |
| Наименьшая точка | Наибольшая точка |
| Конечные структуры | Бесконечные структуры |
| Катаморфизм (свёртка) | Анаморфизм (развёртка) |
| `foldr` | `unfoldr` |

### ⭐ Тип

```haskell
type Coalgebra f a = a -> f a
```

### 💡 Примеры коалгебр

- **Stream**: `seed → (head, tail)`
- **Tree**: `Int → TreeF Int Int` — разворачиваем по индексу
- **Collatz**: `Int → NatF Int` — коалгебра последовательности Коллатца

In [44]:
-- Тип F-коалгебры
type Coalgebra f a = a -> f a

-- Анаморфизм — обобщённый unfold
-- Для конечных структур (используем наш TreeF, ListF)
ana :: Functor f => Coalgebra f a -> a -> Fix f
ana coalg seed = Fix (fmap (ana coalg) (coalg seed))

-- ============================================================
-- Пример 1: Строим списки через коалгебру
-- ============================================================

-- Коалгебра: разворачиваем диапазон [n..m]
rangeCoalg :: (Int, Int) -> ListF Int (Int, Int)
rangeCoalg (n, m)
  | n > m     = NilF
  | otherwise = ConsF n (n+1, m)

-- Коалгебра: итерация — seed = (count, value)
type IterSeed b = (Int, b)

iterCoalg :: (b -> b) -> IterSeed b -> ListF b (IterSeed b)
iterCoalg _ (0, _)    = NilF
iterCoalg f (n, seed) = ConsF seed (n-1, f seed)

-- Коалгебра: Фибоначчи
fibCoalg :: (Integer, Integer, Int) -> ListF Integer (Integer, Integer, Int)
fibCoalg (_, _, 0) = NilF
fibCoalg (a, b, n) = ConsF a (b, a+b, n-1)

buildRange :: Int -> Int -> [Int]
buildRange n m = toList (ana rangeCoalg (n, m))

buildFibs :: Int -> [Integer]
buildFibs n = toList (ana fibCoalg (0, 1, n))

buildIter :: (b -> b) -> Int -> b -> [b]
buildIter f n seed = toList (ana (iterCoalg f) (n, seed))

main6 :: IO ()
main6 = do
  putHeader "Анаморфизм для List (коалгебры)"

  putSub "rangeCoalg: [3..10]"
  putStrLn $ show (buildRange 3 10)

  putSub "fibCoalg: первые 10 Фибоначчи"
  putStrLn $ show (buildFibs 10)

  putSub "iterCoalg (*2): степени двойки"
  putStrLn $ show (buildIter (*2) 8 (1 :: Int))

  putSub "iterCoalg: степени тройки"
  putStrLn $ show (buildIter (*3) 6 (1 :: Int))

  putSub "Последовательность Коллатца от 27 (прямая рекурсия)"
  let collatz n = if even n then n `div` 2 else 3*n + 1
  let collatzSeq start = if start == 1 then [1]
                          else start : collatzSeq (collatz start)
  putStrLn $ "Длина цепи: " ++ show (length (collatzSeq (27 :: Int)))
  putStrLn $ "Первые 10: " ++ show (take 10 (collatzSeq (27 :: Int)))

main6

Line 45: Use print
Found:
putStrLn $ show (buildRange 3 10)
Why not:
print (buildRange 3 10)Line 48: Use print
Found:
putStrLn $ show (buildFibs 10)
Why not:
print (buildFibs 10)Line 51: Use print
Found:
putStrLn $ show (buildIter (* 2) 8 (1 :: Int))
Why not:
print (buildIter (* 2) 8 (1 :: Int))Line 54: Use print
Found:
putStrLn $ show (buildIter (* 3) 6 (1 :: Int))
Why not:
print (buildIter (* 3) 6 (1 :: Int))

= Анаморфизм для List (коалгебры) =

--- rangeCoalg: [3..10] ---
[3,4,5,6,7,8,9,10]
--- fibCoalg: первые 10 Фибоначчи ---
[0,1,1,2,3,5,8,13,21,34]
--- iterCoalg (*2): степени двойки ---
[1,2,4,8,16,32,64,128]
--- iterCoalg: степени тройки ---
[1,3,9,27,81,243]
--- Последовательность Коллатца от 27 (прямая рекурсия) ---
Длина цепи: 112
Первые 10: [27,82,41,124,62,31,94,47,142,71]

In [45]:
-- ============================================================
-- Пример 2: Строим деревья через коалгебру
-- ============================================================

-- Коалгебра для дерева: BST-вставка диапазона
bstCoalg :: (Ord a) => [a] -> TreeF a [a]
bstCoalg [] = LeafF
bstCoalg xs =
  let mid = length xs `div` 2
      (ls, pivot:rs) = splitAt mid (sort xs)
  in NodeF ls pivot rs
  where
    sort = Data.List.sort

buildBST :: Ord a => [a] -> Fix (TreeF a)
buildBST = ana bstCoalg

-- Коалгебра для полного бинарного дерева по глубине
completeTreeCoalg :: Int -> TreeF Int Int
completeTreeCoalg 0 = LeafF
completeTreeCoalg n = NodeF (n-1) n (n-1)

-- Коалгебра Seg-дерева (структура для range-запросов)
-- seed = (lo, hi, values)
segTreeCoalg :: ([Int], Int, Int) -> TreeF Int ([Int], Int, Int)
segTreeCoalg (_, lo, hi)
  | lo == hi  = LeafF
  | otherwise = NodeF ([], lo, mid) (hi - lo + 1) ([], mid+1, hi)
  where mid = (lo + hi) `div` 2

main7 :: IO ()
main7 = do
  putHeader "Анаморфизм для Tree (коалгебры)"

  putSub "BST из [1..7]"
  let bst = buildBST [1..7 :: Int]
  putStrLn $ "Структура: " ++ showTree bst
  putStrLn $ "In-order:  " ++ show (treeInOrder bst)

  putSub "BST из случайных данных"
  let bst2 = buildBST [5,2,8,1,3,7,9 :: Int]
  putStrLn $ "Структура: " ++ showTree bst2
  putStrLn $ "In-order (отсортированный): " ++ show (treeInOrder bst2)

  putSub "Полное дерево глубины 3"
  let ct = ana completeTreeCoalg (3 :: Int)
  putStrLn $ "Глубина: " ++ show (treeDepth ct)
  putStrLn $ "Структура: " ++ showTree ct

main7

= Анаморфизм для Tree (коалгебры) =

--- BST из [1..7] ---
Структура: (((. 1 .) 2 (. 3 .)) 4 ((. 5 .) 6 (. 7 .)))
In-order:  [1,2,3,4,5,6,7]
--- BST из случайных данных ---
Структура: (((. 1 .) 2 (. 3 .)) 5 ((. 7 .) 8 (. 9 .)))
In-order (отсортированный): [1,2,3,5,7,8,9]
--- Полное дерево глубины 3 ---
Глубина: 3
Структура: (((. 1 .) 2 (. 1 .)) 3 ((. 1 .) 2 (. 1 .)))

## 4️⃣ Гиломорфизм: build + fold

### 📐 Теория

**Гиломорфизм** = анаморфизм + катаморфизм.

```haskell
hylo :: Functor f => Algebra f b -> Coalgebra f a -> a -> b
hylo alg coalg = cata alg . ana coalg
-- Или напрямую (без создания Fix):
hylo alg coalg = alg . fmap (hylo alg coalg) . coalg
```

### ⭐ Ключевое преимущество

При слиянии (fusion) компилятор может оптимизировать `hylo` так, что промежуточная структура **никогда не создаётся в памяти** — это называется **deforestation**.

### 💡 Примеры

- **Быстрая сортировка**: разбить на части (коалгебра) → собрать (алгебра)
- **Вычисление выражений**: распарсить (коалгебра) → вычислить (алгебра)
- **Факториал**: отсчёт вниз (коалгебра) → перемножить (алгебра)
- **Сортировка слиянием**: разбить пополам (коалгебра) → смержить (алгебра)

### 🔭 Категориальная перспектива

Гиломорфизм — это **рефлексивный объект**: начальная алгебра и терминальная коалгебра совпадают для строгих функторов.

In [46]:
-- Гиломорфизм
hylo :: Functor f => Algebra f b -> Coalgebra f a -> a -> b
hylo alg coalg = alg . fmap (hylo alg coalg) . coalg

-- ============================================================
-- Пример 1: Факториал через NatF
-- ============================================================
-- Коалгебра: n -> SuccF (n-1) если n > 0, иначе ZeroF
-- Алгебра: ZeroF -> 1, SuccF n -> n+1 ... нет, нам нужен другой трюк

-- Строим список [n, n-1, ..., 1] через коалгебру, потом перемножаем
factCoalg :: Int -> ListF Int Int
factCoalg 0 = NilF
factCoalg n = ConsF n (n - 1)

factAlg :: Algebra (ListF Int) Integer
factAlg NilF        = 1
factAlg (ConsF x r) = fromIntegral x * r

factorial :: Int -> Integer
factorial = hylo factAlg factCoalg

-- ============================================================
-- Пример 2: Быстрая сортировка через TreeF
-- ============================================================
-- Коалгебра: делим по pivot
qsortCoalg :: Ord a => [a] -> TreeF a [a]
qsortCoalg []     = LeafF
qsortCoalg (x:xs) = NodeF smaller x larger
  where
    smaller = filter (< x) xs
    larger  = filter (>= x) xs

-- Алгебра: собираем in-order
qsortAlg :: Algebra (TreeF a) [a]
qsortAlg LeafF         = []
qsortAlg (NodeF l x r) = l ++ [x] ++ r

qsort :: Ord a => [a] -> [a]
qsort = hylo qsortAlg qsortCoalg

-- ============================================================
-- Пример 3: Fibonacci через список пар
-- ============================================================
-- Эффективный Fib через hylo: O(n) без мемоизации
-- ============================================================
-- Пример 3: Числа Фибоначчи через анаморфизм (коалгебру)
-- ============================================================
-- Коалгебра: (a, b, n) -> элемент + следующий seed
fibSeqCoalg :: (Integer, Integer, Int) -> ListF Integer (Integer, Integer, Int)
fibSeqCoalg (_, _, 0) = NilF
fibSeqCoalg (a, b, n) = ConsF a (b, a+b, n-1)

buildFibSeq :: Int -> [Integer]
buildFibSeq n = toList (ana fibSeqCoalg (0, 1, n))

-- fib(n) = n-й элемент последовательности
fibN :: Int -> Integer
fibN 0 = 0
fibN 1 = 1
fibN n = buildFibSeq n !! (n-1)

-- ============================================================
-- Пример 4: Сортировка слиянием через Tree
-- ============================================================
-- Коалгебра: разбить список пополам
msortCoalg :: [a] -> TreeF a [a]
msortCoalg []  = LeafF
msortCoalg [x] = NodeF [] x []
msortCoalg xs  =
  let mid     = length xs `div` 2
      (l, r)  = splitAt mid xs
      (pivot:rest) = l
  in NodeF rest pivot r

-- Алгебра: слить два отсортированных списка
mergeAlg :: Ord a => Algebra (TreeF a) [a]
mergeAlg LeafF = []
mergeAlg (NodeF l x r) = mergeSorted l (x : r)
  where
    mergeSorted [] ys         = ys
    mergeSorted xs []         = xs
    mergeSorted (a:as) (b:bs)
      | a <= b    = a : mergeSorted as (b:bs)
      | otherwise = b : mergeSorted (a:as) bs

mergeSort :: Ord a => [a] -> [a]
mergeSort = hylo mergeAlg msortCoalg

main8 :: IO ()
main8 = do
  putHeader "Гиломорфизмы"

  putSub "Факториал"
  mapM_ (\n -> putStrLn $ show n ++ "! = " ++ show (factorial n)) [0,1,5,10,12]

  putSub "Быстрая сортировка (qsort через hylo)"
  let xs = [3,1,4,1,5,9,2,6,5,3 :: Int]
  putStrLn $ "unsorted: " ++ show xs
  putStrLn $ "sorted:   " ++ show (qsort xs)

  putSub "Числа Фибоначчи через ana"
  putStrLn $ "fib(10) = " ++ show (fibN 10)
  putStrLn $ "fib(20) = " ++ show (fibN 20)
  putStrLn $ "[fib(0..9)] = " ++ show (map fibN [0..9])

  putSub "Сортировка слиянием"
  let ys = [8,3,5,1,7,2,4,6 :: Int]
  putStrLn $ "unsorted: " ++ show ys
  putStrLn $ "sorted:   " ++ show (mergeSort ys)

main8

= Гиломорфизмы =

--- Факториал ---
0! = 1
1! = 1
5! = 120
10! = 3628800
12! = 479001600
--- Быстрая сортировка (qsort через hylo) ---
unsorted: [3,1,4,1,5,9,2,6,5,3]
sorted:   [1,1,2,3,3,4,5,5,6,9]
--- Числа Фибоначчи через ana ---
fib(10) = 34
fib(20) = 4181
[fib(0..9)] = [0,1,1,1,2,3,5,8,13,21]
--- Сортировка слиянием ---
unsorted: [8,3,5,1,7,2,4,6]
sorted:   [3,5,1,8,2,7,4,6]

## 5️⃣ Параморфизм: свёртка с оригиналом

### 📐 Теория

**Параморфизм** — расширение катаморфизма, где на каждом шаге доступна не только **результат рекурсии**, но и **оригинальная подструктура**.

Катаморфизм:
```haskell
cata :: (f a -> a) -> Fix f -> a
```

Параморфизм:
```haskell
para :: Functor f => (f (Fix f, a) -> a) -> Fix f -> a
```

### ⭐ Разница с катаморфизмом

| | Катаморфизм | Параморфизм |
|--|-------------|-------------|
| Алгебра | `f a → a` | `f (Fix f, a) → a` |
| Доступ к подструктуре | ❌ только результат | ✅ и результат, и оригинал |
| Пример | `sum` | `tails` |

### 💡 Когда нужен параморфизм

- Когда нужны «суффиксы» / «хвосты» исходной структуры
- Вычисление `init`, `tails`, `suffixes`
- Алгоритмы, где нужно «смотреть вперёд» в оригинал

### 🔭 Категориальная перспектива

Параморфизм — это морфизм из начальной алгебры в **произведение** `Fix f × a`,
реализованный через проекцию.

In [47]:
-- Параморфизм
para :: Functor f => (f (Fix f, a) -> a) -> Fix f -> a
para alg t = alg (fmap (\s -> (s, para alg s)) (unFix t))

-- ============================================================
-- Пример 1: tails через параморфизм на списках
-- ============================================================
-- tails [1,2,3] = [[1,2,3],[2,3],[3],[]]
tailsAlg :: ListF a (Fix (ListF a), [[a]]) -> [[a]]
tailsAlg NilF              = [[]]
tailsAlg (ConsF _ (t, ts)) = toList t : ts

myTails :: Fix (ListF a) -> [[a]]
myTails = para tailsAlg

-- ============================================================
-- Пример 2: suffixes (все суффиксы как Fix-структуры)
-- ============================================================
suffixesAlg :: ListF a (Fix (ListF a), [Fix (ListF a)]) -> [Fix (ListF a)]
suffixesAlg NilF               = [nilL]
suffixesAlg (ConsF _ (t, rest)) = t : rest

mySuffixes :: Fix (ListF a) -> [Fix (ListF a)]
mySuffixes = para suffixesAlg

-- ============================================================
-- Пример 3: init через параморфизм (удалить последний элемент)
-- ============================================================
myInit :: Fix (ListF a) -> [a]
myInit = para alg
  where
    alg NilF                = []
    alg (ConsF x (rest, _)) = case unFix rest of
      NilF -> []         -- x был последним, не добавляем
      _    -> x : myInit rest

-- ============================================================
-- Пример 4: para для натуральных чисел — Nat-рекурсор
-- ============================================================
natPara :: b -> (Fix NatF -> b -> b) -> Fix NatF -> b
natPara z g = para alg
  where
    alg ZeroF             = z
    alg (SuccF (orig, r)) = g orig r

-- Факториал через примитивную рекурсию
factNat :: Fix NatF -> Integer
factNat = natPara 1 (\n r -> fromIntegral (natVal n + 1) * r)

main9 :: IO ()
main9 = do
  putHeader "Параморфизмы"

  putSub "tails через параморфизм"
  let lst = fromList [1,2,3,4 :: Int]
  putStrLn $ "tails [1,2,3,4] = " ++ show (myTails lst)

  putSub "init через параморфизм"
  putStrLn $ "init [1,2,3,4,5] = " ++ show (myInit (fromList [1..5 :: Int]))

  putSub "suffixes (первые 3 суффикса)"
  let sfxs = mySuffixes lst
  putStrLn $ "Количество суффиксов: " ++ show (length sfxs)
  mapM_ (\s -> putStrLn $ "  " ++ show (toList s)) sfxs

  putSub "Факториал через natPara (примитивная рекурсия)"
  mapM_ (\n -> putStrLn $ show (natVal n) ++ "! = " ++ show (factNat n))
    [zero, one, two, three, four, five]

main9

= Параморфизмы =

--- tails через параморфизм ---
tails [1,2,3,4] = [[2,3,4],[3,4],[4],[],[]]
--- init через параморфизм ---
init [1,2,3,4,5] = [1,2,3,4]
--- suffixes (первые 3 суффикса) ---
Количество суффиксов: 5
  [2,3,4]
  [3,4]
  [4]
  []
  []
--- Факториал через natPara (примитивная рекурсия) ---
0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120

## 6️⃣ Апоморфизм: развёртка с «прыжком»

### 📐 Теория

**Апоморфизм** — двойник параморфизма: это анаморфизм, где на каждом шаге можно **либо продолжить развёртку**, либо **прыгнуть** к готовой структуре.

Анаморфизм:
```haskell
ana :: (a -> f a) -> a -> Fix f
```

Апоморфизм:
```haskell
apo :: Functor f => (a -> f (Either (Fix f) a)) -> a -> Fix f
```

- `Right seed` → продолжить развёртку с новым seed
- `Left structure` → встроить готовую структуру (остановиться)

### 💡 Когда нужен апоморфизм

- Когда хочется «встроить» уже готовый кусок структуры без обхода
- Вставка в BST (нашли позицию → прыгаем)
- Конкатенация списков (второй список — уже готов)
- Алгоритмы с «ранней остановкой» при построении

### 🔭 Категориальная перспектива

Апоморфизм — это ко-рекурсор, двойник примитивной рекурсии.
Использует копроизведение (Either) вместо произведения.

In [48]:
-- Апоморфизм
apo :: Functor f => (a -> f (Either (Fix f) a)) -> a -> Fix f
apo coalg seed = Fix (fmap step (coalg seed))
  where
    step (Left  t) = t
    step (Right s) = apo coalg s

-- ============================================================
-- Пример 1: Конкатенация через апоморфизм
-- ============================================================
-- Идея: строим первый список, в конце «прыгаем» ко второму
appendCoalg :: ([a], [a]) -> ListF a (Either (Fix (ListF a)) ([a], [a]))
appendCoalg ([], ys)   = case ys of
  []     -> NilF
  (y:yt) -> ConsF y (Right ([], yt))  -- продолжаем со вторым
appendCoalg (x:xs, ys) = ConsF x (Right (xs, ys))

-- Версия с прыжком: как только первый кончился, встраиваем второй сразу
appendFast :: Fix (ListF a) -> Fix (ListF a) -> Fix (ListF a)
appendFast xs ys = apo coalg (toList xs)
  where
    coalg []     = case unFix ys of
      NilF       -> NilF
      ConsF h t  -> ConsF h (Left t)  -- прыгаем в готовую структуру
    coalg (x:rest) = ConsF x (Right rest)

-- ============================================================
-- Пример 2: Вставка в отсортированный список
-- ============================================================
insertSortedCoalg :: Ord a => (a, [a]) -> ListF a (Either (Fix (ListF a)) (a, [a]))
insertSortedCoalg (x, [])   = ConsF x (Left nilL)
insertSortedCoalg (x, y:ys)
  | x <= y    = ConsF x (Left (fromList (y:ys)))  -- найдено место — прыгаем
  | otherwise = ConsF y (Right (x, ys))            -- ещё не место — продолжаем

insertSorted :: Ord a => a -> Fix (ListF a) -> Fix (ListF a)
insertSorted x ys = apo (insertSortedCoalg) (x, toList ys)

-- ============================================================
-- Пример 3: Развёртка с условной остановкой
-- ============================================================
-- Строим список пока условие выполнено, потом вставляем «стоп»
takeWhileCoalg :: (a -> Bool) -> (a -> a) -> (a, Int) -> ListF a (Either (Fix (ListF a)) (a, Int))
takeWhileCoalg p f (x, n)
  | n <= 0 || not (p x) = NilF
  | otherwise = ConsF x (Right (f x, n-1))

-- Пример: все чётные начиная с n, пока < 20
evensBelowCoalg :: Int -> ListF Int (Either (Fix (ListF Int)) Int)
evensBelowCoalg n
  | n >= 20   = ConsF 20 (Left nilL)  -- достигли предела — встраиваем стоп
  | even n    = ConsF n  (Right (n+2))
  | otherwise = ConsF (n+1) (Right (n+3))

evensBelow20 :: Int -> [Int]
evensBelow20 start = toList (apo evensBelowCoalg start)

main10 :: IO ()
main10 = do
  putHeader "Апоморфизмы"

  putSub "Конкатенация через apo"
  let xs = fromList [1,2,3 :: Int]
      ys = fromList [4,5,6 :: Int]
  putStrLn $ "[1,2,3] ++ [4,5,6] = " ++ show (toList (appendFast xs ys))

  putSub "Вставка в отсортированный список"
  let sorted = fromList [1,3,5,7,9 :: Int]
  putStrLn $ "insert 4 [1,3,5,7,9] = " ++ show (toList (insertSorted 4 sorted))
  putStrLn $ "insert 0 [1,3,5,7,9] = " ++ show (toList (insertSorted 0 sorted))
  putStrLn $ "insert 10 [1,3,5,7,9] = " ++ show (toList (insertSorted 10 sorted))

  putSub "Чётные числа < 20 через apo"
  putStrLn $ "от 2: " ++ show (evensBelow20 2)
  putStrLn $ "от 6: " ++ show (evensBelow20 6)

main10

Line 37: Redundant bracket
Found:
(insertSortedCoalg)
Why not:
insertSortedCoalg

= Апоморфизмы =

--- Конкатенация через apo ---
[1,2,3] ++ [4,5,6] = [1,2,3,4,5,6]
--- Вставка в отсортированный список ---
insert 4 [1,3,5,7,9] = [1,3,4,5,7,9]
insert 0 [1,3,5,7,9] = [0,1,3,5,7,9]
insert 10 [1,3,5,7,9] = [1,3,5,7,9,10]
--- Чётные числа < 20 через apo ---
от 2: [2,4,6,8,10,12,14,16,18,20]
от 6: [6,8,10,12,14,16,18,20]

## 7️⃣ Гистоморфизм: свёртка с историей

### 📐 Теория

**Гистоморфизм** — это катаморфизм, где алгебра имеет доступ не только к результатам рекурсии, но и ко **всей истории** вычислений на подструктурах.

Это моделируется через **кофрейм (Cofree)**:
```haskell
data Cofree f a = a :< f (Cofree f a)
```

- Голова `a` — кешированный результат
- Хвост `f (Cofree f a)` — подструктуры с их кешированными результатами

Гистоморфизм:
```haskell
histo :: Functor f => (f (Cofree f a) -> a) -> Fix f -> a
```

### ⭐ Сравнение схем

| Схема | Доступ к прошлому |
|-------|------------------|
| cata | Только финальный результат подзадачи |
| para | Результат + оригинальная структура |
| histo | Полная «история» (все промежуточные результаты) |

### 💡 Классический пример

**Числа Фибоначчи** — нужно `fib(n-1)` и `fib(n-2)`. Это требует истории двух последних шагов, что идеально для гистоморфизма.

### 🔭 Связь с динамическим программированием

Гистоморфизм — это **структурное динамическое программирование**: история — это таблица DP, встроенная в структуру обхода.

In [49]:
-- Кофрейм (Cofree коалгебра)
data Cofree f a = a :< f (Cofree f a)

infixr 5 :<

-- Доступ к кешированному значению
extract :: Cofree f a -> a
extract (a :< _) = a

-- Доступ к подструктурам
unwrap :: Cofree f a -> f (Cofree f a)
unwrap (_ :< fs) = fs

-- Гистоморфизм
histo :: Functor f => (f (Cofree f a) -> a) -> Fix f -> a
histo alg = extract . worker
  where
    worker t = let fs  = fmap worker (unFix t)
                   res = alg fs
               in res :< fs

-- ============================================================
-- Пример 1: Fibonacci через гистоморфизм
-- ============================================================
-- NatF: ZeroF | SuccF r
-- На каждом шаге: SuccF даёт нам историю предыдущих вычислений

fibHisto :: Fix NatF -> Integer
fibHisto = histo alg
  where
    alg ZeroF        = 0
    alg (SuccF hist) =
      let n_1 = extract hist  -- fib(n-1)
          n_2 = case unwrap hist of
                  ZeroF       -> 0  -- fib(-1) = 0 (база)
                  SuccF hist' -> extract hist'  -- fib(n-2)
      in n_1 + n_2

-- ============================================================
-- Пример 2: Числа Трибоначчи (требует 3 предыдущих шага)
-- ============================================================
tribHisto :: Fix NatF -> Integer
tribHisto = histo alg
  where
    alg ZeroF        = 0
    alg (SuccF h1)   =
      let t1 = extract h1
          t2 = case unwrap h1 of
                 ZeroF  -> 0
                 SuccF h2 -> extract h2
          t3 = case unwrap h1 of
                 ZeroF  -> 0
                 SuccF h2 -> case unwrap h2 of
                               ZeroF  -> 1  -- trib(1) = 1
                               SuccF h3 -> extract h3
      in t1 + t2 + t3

-- ============================================================
-- Пример 3: Числа Каталана через гистоморфизм
-- ============================================================
-- C(0) = 1, C(n+1) = sum_{i=0}^{n} C(i) * C(n-i)
-- Нужна вся история C(0), ..., C(n)

-- Сохраняем историю в списке
catalanHisto :: Fix NatF -> Integer
catalanHisto = histo alg
  where
    alg ZeroF        = 1  -- C(0) = 1
    alg (SuccF hist) =
      let history = collectHistory hist
      in sum [history !! i * history !! (length history - 1 - i)
             | i <- [0..length history - 1]]

    collectHistory :: Cofree NatF Integer -> [Integer]
    collectHistory h = extract h : case unwrap h of
      ZeroF     -> []
      SuccF h'  -> collectHistory h'

main11 :: IO ()
main11 = do
  putHeader "Гистоморфизмы"

  putSub "Fibonacci через histo (O(n), без мемоизации)"
  putStrLn $ "fib(0..10) = " ++ show (map (fibHisto . buildNat) [0..10])
  putStrLn $ "fib(20) = " ++ show (fibHisto (buildNat 20))
  putStrLn $ "fib(30) = " ++ show (fibHisto (buildNat 30))

  putSub "Трибоначчи через histo"
  putStrLn $ "trib(0..10) = " ++ show (map (tribHisto . buildNat) [0..10])

  putSub "Числа Каталана через histo"
  putStrLn $ "catalan(0..8) = " ++ show (map (catalanHisto . buildNat) [0..8])
  -- C(n) = 1,1,2,5,14,42,132,429,1430

  where
    buildNat :: Int -> Fix NatF
    buildNat 0 = Fix ZeroF
    buildNat n = Fix (SuccF (buildNat (n-1)))

main11

= Гистоморфизмы =

--- Fibonacci через histo (O(n), без мемоизации) ---
fib(0..10) = [0,0,0,0,0,0,0,0,0,0,0]
fib(20) = 0
fib(30) = 0
--- Трибоначчи через histo ---
trib(0..10) = [0,0,1,1,2,4,7,13,24,44,81]
--- Числа Каталана через histo ---
catalan(0..8) = [1,1,2,5,14,42,132,429,1430]

## 8️⃣ Футуморфизм: развёртка с предсказаниями

### 📐 Теория

**Футуморфизм** — двойник гистоморфизма: это анаморфизм, где коалгебра может производить не просто `seed`, а **целый кусок будущей структуры**.

Моделируется через **свободный монад (Free)**:
```haskell
data Free f a = Pure a | Free (f (Free f a))
```

Футуморфизм:
```haskell
futu :: Functor f => (a -> f (Free f a)) -> a -> Fix f
```

- `Pure seed` → обычное продолжение (как в анаморфизме)
- `Free structure` → «предсказание» — вставить готовый кусок структуры

### ⭐ Дуальность

| Концепт | Прошлое (история) | Будущее (предсказание) |
|---------|-------------------|------------------------|
| Структура | Cofree f a | Free f a |
| Схема рекурсии | Histomorphism | Futumorphism |
| Направление | Свёртка (cata) | Развёртка (ana) |

### 💡 Применения

- Строить структуры с «ленивым» вычислением нескольких шагов за раз
- Генерация кода с «блоками» предопределённой структуры
- Потоки событий с группировкой

In [50]:
-- Свободный монад (Free)
data Free f a = Pure a | Free (f (Free f a)) deriving (Functor)

-- Футуморфизм
futu :: Functor f => (a -> f (Free f a)) -> a -> Fix f
futu coalg = Fix . fmap interpret . coalg
  where
    interpret (Pure seed)     = futu coalg seed
    interpret (Free structure) = Fix (fmap interpret structure)

-- ============================================================
-- Пример 1: Строим список с "пакетной" вставкой
-- ============================================================
-- Обычный ana вставляет по одному элементу.
-- futu может вставить "блок" сразу.

-- Коалгебра: вставляем числа, но при кратных 3 — вставляем сразу тройку
batchCoalg :: Int -> ListF Int (Free (ListF Int) Int)
batchCoalg 0 = NilF
batchCoalg n
  | n `mod` 3 == 0 =
      -- Вставляем n, n*10, n*100 сразу за один шаг коалгебры
      ConsF n (Free (ConsF (n*10) (Free (ConsF (n*100) (Pure (n-1))))))
  | otherwise = ConsF n (Pure (n-1))

buildBatchList :: Int -> [Int]
buildBatchList = toList . futu batchCoalg

-- ============================================================
-- Пример 2: Развёртка дерева с "готовыми" поддеревьями
-- ============================================================
-- Строим дерево поиска, но при нахождении диапазона [a..b]
-- сразу вставляем сбалансированное поддерево

treeWithPrediction :: [Int] -> Fix (TreeF Int)
treeWithPrediction = futu treeCoalg
  where
    treeCoalg [] = LeafF
    treeCoalg [x] = NodeF (Pure []) x (Pure [])
    treeCoalg xs  =
      let mid   = length xs `div` 2
          (l, pivot:r) = splitAt mid xs
      in NodeF (Pure l) pivot (Pure r)

-- ============================================================
-- Пример 3: Последовательность с "перепрыгиванием"
-- ============================================================
-- Строим список, каждые k шагов "предсказываем" несколько следующих
strideCoalg :: Int -> (Int, Int) -> ListF Int (Free (ListF Int) (Int, Int))
strideCoalg stride (n, limit)
  | n > limit = NilF
  | n `mod` stride == 0 =
      -- При кратном stride — генерируем stride элементов вперёд сразу
      let rest = take (stride-1) (zip [n+1..] (repeat limit))
          mkChain []              = Pure (n + stride, limit)
          mkChain ((x,lim):more) = Free (ConsF x (mkChain more))
      in ConsF n (mkChain rest)
  | otherwise = ConsF n (Pure (n+1, limit))

buildStride :: Int -> Int -> [Int]
buildStride stride limit = toList (futu (strideCoalg stride) (1, limit))

main12 :: IO ()
main12 = do
  putHeader "Футуморфизмы"

  putSub "Список с пакетной вставкой (кратные 3)"
  putStrLn $ "buildBatchList 6 = " ++ show (buildBatchList 6)
  -- При n=6: вставит 6, 60, 600, затем 5, затем 4, 40, 400, затем 3, 30, 300, ...

  putSub "Дерево через futu"
  let t = treeWithPrediction [1..7]
  putStrLn $ "Структура: " ++ showTree t
  putStrLn $ "In-order:  " ++ show (treeInOrder t)

  putSub "Stride-последовательность (stride=3, до 15)"
  putStrLn $ show (buildStride 3 15)

main12

Line 54: Use map with tuple-section
Found:
zip [n + 1 .. ] (repeat limit)
Why not:
map (, limit) [n + 1 .. ]Line 77: Use print
Found:
putStrLn $ show (buildStride 3 15)
Why not:
print (buildStride 3 15)

= Футуморфизмы =

--- Список с пакетной вставкой (кратные 3) ---
buildBatchList 6 = [6,60,600,5,4,3,30,300,2,1]
--- Дерево через futu ---
Структура: (((. 1 .) 2 (. 3 .)) 4 ((. 5 .) 6 (. 7 .)))
In-order:  [1,2,3,4,5,6,7]
--- Stride-последовательность (stride=3, до 15) ---
[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17]

## 9️⃣ Синтез: Язык выражений

### 📐 Практический пример

Рассмотрим, как разные схемы рекурсии применяются к **языку арифметических выражений**.

F-функтор для выражений:
```haskell
data ExprF r
  = NumF  Double
  | VarF  String
  | AddF  r r
  | MulF  r r
  | NegF  r
  | IfF   r r r   -- if condition then else
  | LetF  String r r  -- let x = e1 in e2
```

Это демонстрирует:
- **cata**: вычисление / трансляция
- **para**: анализ с доступом к AST
- **histo**: оптимизация с историей (constant folding)
- **ana**: парсинг / генерация кода

### 💡 Реальное применение

Компиляторы (GHC, LLVM IR, Roslyn) внутренне используют именно эти паттерны для трансформации AST.

In [51]:
import qualified Data.Map as Map
-- ============================================================
-- F-функтор для выражений
-- ============================================================
data ExprF r
  = NumF  Double
  | VarF  String
  | AddF  r r
  | MulF  r r
  | NegF  r
  | DivF  r r
  deriving (Show, Functor)

type Expr = Fix ExprF

-- Умные конструкторы
num :: Double -> Expr
num = Fix . NumF

var :: String -> Expr
var = Fix . VarF

add :: Expr -> Expr -> Expr
add l r = Fix (AddF l r)

mul :: Expr -> Expr -> Expr
mul l r = Fix (MulF l r)

neg :: Expr -> Expr
neg = Fix . NegF

divE :: Expr -> Expr -> Expr
divE l r = Fix (DivF l r)

type Env = Map.Map String Double

-- ============================================================
-- Катаморфизм: вычислить выражение
-- ============================================================
evalAlg :: Env -> Algebra ExprF (Either String Double)
evalAlg _ (NumF n)     = Right n
evalAlg e (VarF v)     = maybe (Left $ "Неизвестная переменная: " ++ v) Right (Map.lookup v e)
evalAlg _ (AddF l r)   = (+) <$> l <*> r
evalAlg _ (MulF l r)   = (*) <$> l <*> r
evalAlg _ (NegF x)     = negate <$> x
evalAlg _ (DivF l r)   = do
  lv <- l; rv <- r
  if rv == 0 then Left "Деление на ноль" else Right (lv / rv)

eval :: Env -> Expr -> Either String Double
eval env = cata (evalAlg env)

-- ============================================================
-- Катаморфизм: красивая печать
-- ============================================================
prettyAlg :: Algebra ExprF String
prettyAlg (NumF n)   = if n == fromIntegral (round n :: Int)
                        then show (round n :: Int) else show n
prettyAlg (VarF v)   = v
prettyAlg (AddF l r) = "(" ++ l ++ " + " ++ r ++ ")"
prettyAlg (MulF l r) = "(" ++ l ++ " * " ++ r ++ ")"
prettyAlg (NegF x)   = "(-" ++ x ++ ")"
prettyAlg (DivF l r) = "(" ++ l ++ " / " ++ r ++ ")"

pretty :: Expr -> String
pretty = cata prettyAlg

-- ============================================================
-- Катаморфизм: подсчёт операций
-- ============================================================
data Stats = Stats { numOps :: Int, numVars :: Int, numNums :: Int } deriving Show

statsAlg :: Algebra ExprF Stats
statsAlg (NumF _)   = Stats 0 0 1
statsAlg (VarF _)   = Stats 0 1 0
statsAlg (AddF l r) = Stats (numOps l + numOps r + 1) (numVars l + numVars r) (numNums l + numNums r)
statsAlg (MulF l r) = Stats (numOps l + numOps r + 1) (numVars l + numVars r) (numNums l + numNums r)
statsAlg (NegF x)   = Stats (numOps x + 1) (numVars x) (numNums x)
statsAlg (DivF l r) = Stats (numOps l + numOps r + 1) (numVars l + numVars r) (numNums l + numNums r)

countStats :: Expr -> Stats
countStats = cata statsAlg

main13 :: IO ()
main13 = do
  putHeader "Катаморфизмы для языка выражений"

  -- Выражение: (x + 2) * (y - 3) / (x + y)
  let expr1 = divE (mul (add (var "x") (num 2)) (add (var "y") (neg (num 3)))) (add (var "x") (var "y"))
  let env   = Map.fromList [("x", 5.0), ("y", 7.0)]

  putSub "Красивая печать"
  putStrLn $ pretty expr1

  putSub "Вычисление при x=5, y=7"
  putStrLn $ show (eval env expr1)

  putSub "Ошибка: деление на ноль"
  let expr2 = divE (num 10) (add (var "a") (num 0))
  let env2  = Map.fromList [("a", 0.0)]
  putStrLn $ show (eval env2 expr2)

  putSub "Статистика AST"
  let s = countStats expr1
  putStrLn $ "Операций: " ++ show (numOps s)
  putStrLn $ "Переменных: " ++ show (numVars s)
  putStrLn $ "Констант: " ++ show (numNums s)

main13

Line 96: Use print
Found:
putStrLn $ show (eval env expr1)
Why not:
print (eval env expr1)Line 101: Use print
Found:
putStrLn $ show (eval env2 expr2)
Why not:
print (eval env2 expr2)

= Катаморфизмы для языка выражений =

--- Красивая печать ---
(((x + 2) * (y + (-3))) / (x + y))
--- Вычисление при x=5, y=7 ---
Right 2.3333333333333335
--- Ошибка: деление на ноль ---
Left "\1044\1077\1083\1077\1085\1080\1077 \1085\1072 \1085\1086\1083\1100"
--- Статистика AST ---
Операций: 6
Переменных: 4
Констант: 2

In [52]:
-- ============================================================
-- Параморфизм: оптимизация (constant folding) с доступом к AST
-- ============================================================
-- Если оба аргумента числа — сворачиваем сразу
-- Иначе — сохраняем структуру, но показываем оригинальный узел

simplifyAlg :: ExprF (Expr, Expr) -> Expr
simplifyAlg (NumF n)           = num n
simplifyAlg (VarF v)           = var v
simplifyAlg (NegF (_, e))      = case unFix e of
  NumF n -> num (negate n)
  _      -> neg e
simplifyAlg (AddF (_, l) (_, r)) = case (unFix l, unFix r) of
  (NumF a, NumF b) -> num (a + b)
  (NumF 0, _)      -> r
  (_, NumF 0)      -> l
  _                -> add l r
simplifyAlg (MulF (_, l) (_, r)) = case (unFix l, unFix r) of
  (NumF a, NumF b) -> num (a * b)
  (NumF 1, _)      -> r
  (_, NumF 1)      -> l
  (NumF 0, _)      -> num 0
  (_, NumF 0)      -> num 0
  _                -> mul l r
simplifyAlg (DivF (_, l) (_, r)) = case (unFix l, unFix r) of
  (NumF a, NumF b) | b /= 0 -> num (a / b)
  (_, NumF 1)                -> l
  _                          -> divE l r

simplify :: Expr -> Expr
simplify = para simplifyAlg

main14 :: IO ()
main14 = do
  putHeader "Параморфизм: constant folding"

  putSub "Упрощение выражений"
  let examples =
        [ add (num 3) (num 4)                          -- 3 + 4 = 7
        , mul (num 1) (var "x")                        -- 1 * x = x
        , add (num 0) (mul (num 2) (num 3))            -- 0 + (2*3) = 6
        , neg (num 5)                                  -- -(5) = -5
        , mul (num 2) (add (num 3) (num 4))            -- 2*(3+4) = 14
        , add (var "x") (mul (num 0) (var "y"))        -- x + 0*y = x + 0 = x
        ]
  mapM_ (\e -> putStrLn $ pretty e ++ "  =>  " ++ pretty (simplify e)) examples

main14

= Параморфизм: constant folding =

--- Упрощение выражений ---
(3 + 4)  =>  7
(1 * x)  =>  x
(0 + (2 * 3))  =>  6
(-5)  =>  -5
(2 * (3 + 4))  =>  14
(x + (0 * y))  =>  x

## 🔟 Схемы рекурсии: таблица и законы

### 📐 Полная таблица схем

| Схема | Тип | Инструмент | Аналог |
|-------|-----|-----------|--------|
| **cata** | `(f a -> a) -> Fix f -> a` | Algebra | `foldr` |
| **ana** | `(a -> f a) -> a -> Fix f` | Coalgebra | `unfoldr` |
| **hylo** | `(f b -> b) -> (a -> f a) -> a -> b` | Alg + Coalg | build/fold |
| **para** | `(f (Fix f, a) -> a) -> Fix f -> a` | RAlgebra | примитивная рекурсия |
| **apo** | `(a -> f (Either (Fix f) a)) -> a -> Fix f` | RCoalgebra | примитивная корекурсия |
| **histo** | `(f (Cofree f a) -> a) -> Fix f -> a` | CVAlgebra | DP / мемоизация |
| **futu** | `(a -> f (Free f a)) -> a -> Fix f` | CVCoalgebra | генерация с блоками |

### ⭐ Ключевые законы

**Fusion (слияние)**:
```
h . cata alg = cata alg'  когда  h . alg = alg' . fmap h
```

**Рефлексивность**:
```
cata Fix = id
```

**Гиломорфизм**:
```
hylo alg coalg = cata alg . ana coalg
```

### 💡 Практические рекомендации

1. Начни с **cata** - покрывает 80% случаев
2. Нужен оригинал структуры -> **para**
3. Строишь структуру -> **ana**, нужен прыжок -> **apo**
4. Нужна история предыдущих шагов -> **histo** (DP)
5. И строишь, и сворачиваешь -> **hylo** (deforestation)

### 📐 Как каждая схема строится из предыдущей

![Иерархия схем рекурсии](../diagrams/algebras/recursion_schemes.svg)

**Гиломорфизм** связывает ana и cata:
```
hylo alg coalg = cata alg . ana coalg  -- с оптимизацией: без промежуточной структуры
```

In [53]:
-- =================================================================
-- Дерево гиломорфизма: промежуточная структура
-- =================================================================

-- Гиломорфизм = развернуть коалгеброй, затем свернуть алгеброй
-- Промежуточная структура -- это дерево вычислений

-- Пример: сортировка слиянием через гиломорфизм
-- Коалгебра разбивает список пополам (развертывание = дерево)
-- Алгебра сливает два отсортированных списка (свертывание)

data TreeF a r = LeafF a | NodeF r r deriving (Show, Functor)

type Tree a = Fix (TreeF a)

-- Вспомогательные конструкторы
leaf :: a -> Tree a
leaf x = Fix (LeafF x)

branch :: Tree a -> Tree a -> Tree a
branch l r = Fix (NodeF l r)

-- Показать структуру дерева (промежуточную)
showTree :: Show a => Tree a -> String
showTree = cata alg
  where
    alg (LeafF x)   = "[" ++ show x ++ "]"
    alg (NodeF l r) = "(" ++ l ++ "+" ++ r ++ ")"

-- Коалгебра: разбить список на дерево слияния
splitCoalg :: [Int] -> TreeF Int [Int]
splitCoalg [x] = LeafF x
splitCoalg xs  = let (l, r) = splitAt (length xs `div` 2) xs
                 in NodeF l r

-- Алгебра: слить два отсортированных списка
mergeAlg :: TreeF Int [Int] -> [Int]
mergeAlg (LeafF x)   = [x]
mergeAlg (NodeF l r) = merge2 l r
  where
    merge2 [] ys = ys
    merge2 xs [] = xs
    merge2 (x:xs) (y:ys)
      | x <= y    = x : merge2 xs (y:ys)
      | otherwise = y : merge2 (x:xs) ys

-- Гиломорфизм для сортировки слиянием
mergeSort :: [Int] -> [Int]
mergeSort = hylo mergeAlg splitCoalg

-- Визуализация промежуточного дерева
visualize :: IO ()
visualize = do
  putHeader "Дерево гиломорфизма (промежуточная структура)"
  let xs = [5, 2, 8, 1, 9, 3, 7, 4 :: Int]
  putStrLn $ "Вход: " ++ show xs
  let tree = ana splitCoalg xs
  putStrLn "Промежуточное дерево:"
  putStrLn $ "  " ++ showTree tree
  putStrLn $ "Отсортировано: " ++ show (mergeSort xs)
  let ys = [3, 1, 4, 1, 5 :: Int]
  let treeY = ana splitCoalg ys
  putStrLn ""
  putStrLn $ "Дерево для " ++ show ys ++ ":"
  putStrLn $ "  " ++ showTree treeY
  putStrLn $ "После слияния: " ++ show (mergeSort ys)

visualize

= Дерево гиломорфизма (промежуточная структура) =

Вход: [5,2,8,1,9,3,7,4]
Промежуточное дерево:
  ((([5]+[2])+([8]+[1]))+(([9]+[3])+([7]+[4])))
Отсортировано: [1,2,3,4,5,7,8,9]

Дерево для [3,1,4,1,5]:
  (([3]+[1])+([4]+([1]+[5])))
После слияния: [1,1,3,4,5]

In [54]:
-- Diagram: Hylomorphism = ana then cata (two-phase)
-- Shows: A -[ana/unfold]-> mu F -[cata/fold]-> B
-- with intermediate tree structure

let hyloSvg = "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"440\" height=\"160\">" ++
      "<defs><marker id=\"arr2\" markerWidth=\"10\" markerHeight=\"7\" refX=\"10\" refY=\"3.5\" orient=\"auto\">" ++
      "<polygon points=\"0 0, 10 3.5, 0 7\" fill=\"black\"/></marker></defs>" ++
      -- three boxes: A (seed), muF (tree), B (result)
      "<rect x=\"10\" y=\"62\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#e8ffe8\" stroke=\"#44aa44\" stroke-width=\"1.5\"/>" ++
      "<text x=\"50\" y=\"85\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"16\" font-style=\"italic\">A</text>" ++
      -- middle: tree shape
      "<rect x=\"170\" y=\"40\" width=\"100\" height=\"80\" rx=\"6\" fill=\"#f0f0ff\" stroke=\"#8888cc\" stroke-width=\"1.5\"/>" ++
      "<text x=\"220\" y=\"68\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"13\">&#956;F</text>" ++
      -- tree nodes inside the box
      "<circle cx=\"220\" cy=\"90\" r=\"5\" fill=\"#8888cc\"/>" ++
      "<circle cx=\"200\" cy=\"108\" r=\"4\" fill=\"#aaaadd\"/>" ++
      "<circle cx=\"240\" cy=\"108\" r=\"4\" fill=\"#aaaadd\"/>" ++
      "<line x1=\"220\" y1=\"95\" x2=\"204\" y2=\"104\" stroke=\"#888\" stroke-width=\"1\"/>" ++
      "<line x1=\"220\" y1=\"95\" x2=\"236\" y2=\"104\" stroke=\"#888\" stroke-width=\"1\"/>" ++
      "<text x=\"220\" y=\"128\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"9\" fill=\"#666\">(intermediate)</text>" ++
      "<rect x=\"350\" y=\"62\" width=\"80\" height=\"36\" rx=\"6\" fill=\"#fff0e8\" stroke=\"#cc8844\" stroke-width=\"1.5\"/>" ++
      "<text x=\"390\" y=\"85\" text-anchor=\"middle\" font-family=\"serif\" font-size=\"16\" font-style=\"italic\">B</text>" ++
      -- arrow A -> muF  (ana / coalgebra)
      "<line x1=\"92\" y1=\"80\" x2=\"168\" y2=\"80\" stroke=\"#4488cc\" stroke-width=\"2\" marker-end=\"url(#arr2)\"/>" ++
      "<text x=\"130\" y=\"70\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\" fill=\"#4488cc\">ana / unfold</text>" ++
      "<text x=\"130\" y=\"97\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"9\" fill=\"#888\">(coalgebra)</text>" ++
      -- arrow muF -> B  (cata / algebra)
      "<line x1=\"272\" y1=\"80\" x2=\"348\" y2=\"80\" stroke=\"#cc4444\" stroke-width=\"2\" marker-end=\"url(#arr2)\"/>" ++
      "<text x=\"310\" y=\"70\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"11\" fill=\"#cc4444\">cata / fold</text>" ++
      "<text x=\"310\" y=\"97\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"9\" fill=\"#888\">(algebra)</text>" ++
      -- shortcut arrow A -> B  (hylo)
      "<path d=\"M 50 62 C 50 20, 390 20, 390 62\" stroke=\"#884488\" stroke-width=\"1.5\" fill=\"none\" stroke-dasharray=\"5,3\" marker-end=\"url(#arr2)\"/>" ++
      "<text x=\"220\" y=\"15\" text-anchor=\"middle\" font-family=\"monospace\" font-size=\"12\" fill=\"#884488\">hylo = cata . ana</text>" ++
      "<text x=\"220\" y=\"152\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"11\" fill=\"#555\">Hylomorphism deforests the intermediate tree</text>" ++
      "</svg>"

writeFile "/tmp/hylo_tree.svg" hyloSvg
putStrLn "Hylo diagram saved."

Hylo diagram saved.

In [55]:
:! cp /tmp/hylo_tree.svg /home/jovyan/src/ && echo "copied hylo_tree.svg"

copied hylo_tree.svg

### Диаграмма: Гиломорфизм — промежуточное дерево

![Гиломорфизм: разворачивание и сворачивание через промежуточное дерево](../diagrams/algebras/hylo_tree.svg)

`hylo coalg alg = cata alg . ana coalg`  
DEFORESTATION: GHC оптимизирует `hylo alg coalg = alg . fmap (hylo alg coalg) . coalg` — дерево никогда не строится в памяти.

In [56]:
-- ============================================================
-- Демонстрация законов
-- ============================================================

-- Закон: para выражается через cata
paraViaCata :: Functor f => (f (Fix f, a) -> a) -> Fix f -> a
paraViaCata alg = snd . cata step
  where
    step f = (Fix (fmap fst f), alg f)

-- Проверка
lawCheck :: IO ()
lawCheck = do
  putHeader "Проверка законов схем рекурсии"

  putSub "cata Fix = id"
  let nats = [zero, one, two, three, four, five]
  let allOk = all (\t -> cata natToInt (cata Fix t) == cata natToInt t) nats
  putStrLn $ "Для nat 0..5: " ++ show allOk

  putSub "hylo = cata . ana"
  let testLists = [[3,1,4,1,5],[9,2,6],[1],[],[5,4,3,2,1 :: Int]]
  let check xs = hylo qsortAlg qsortCoalg xs == (cata qsortAlg . ana qsortCoalg) xs
  putStrLn $ "qsort hylo == cata.ana: " ++ show (all check testLists)

  putSub "para == paraViaCata"
  let lst = fromList [1,2,3,4,5 :: Int]
  let t1 = myTails lst
  let t2 = paraViaCata tailsAlg lst
  putStrLn $ "para:        " ++ show t1
  putStrLn $ "paraViaCata: " ++ show t2
  putStrLn $ "Равны: " ++ show (t1 == t2)

lawCheck

= Проверка законов схем рекурсии =

--- cata Fix = id ---
Для nat 0..5: True
--- hylo = cata . ana ---
qsort hylo == cata.ana: True
--- para == paraViaCata ---
para:        [[2,3,4,5],[3,4,5],[4,5],[5],[],[]]
paraViaCata: [[2,3,4,5],[3,4,5],[4,5],[5],[],[]]
Равны: True

## 11. Категориальная перспектива

### 🔭 Теория категорий

**F-алгебра** в категории Hask - это объект A вместе со стрелкой `alpha :: F A -> A`.

Морфизм F-алгебр `(A, alpha) -> (B, beta)` - это `f :: A -> B`, такое что:
```
beta . fmap f = f . alpha
```

Это называется **гомоморфизм F-алгебр**.

### Теорема Ламбека

Если `(mu_F, in_F)` - начальная F-алгебра, то `in_F :: F(mu_F) -> mu_F` является **изоморфизмом**.

Это означает: `mu_F ~= F(mu_F)` - фундаментальное уравнение рекурсивных типов.

### Начальная алгебра = Fix

```
Fix F  ~=  F (Fix F)
Fix { unFix } :: F (Fix F) <-> Fix F
```

Катаморфизм - единственный гомоморфизм из начальной алгебры:
```
cata alg :: Fix F -> A
```

### Коммутативная диаграмма:

```
F(Fix F)  --fmap (cata alg)-->  F A
    |                             |
  Fix                           alg
    |                             |
 Fix F     ---cata alg------->    A
```

### Дуальность

```
F-алгебра              F-коалгебра
(A, F A -> A)          (A, A -> F A)
Начальная алгебра      Терминальная коалгебра
mu_F (наименьшая ТНФ)  nu_F (наибольшая ТНФ)
Конечные деревья       Бесконечные потоки
cata (foldr)           ana (unfoldr)
```

### Связь с теорией доменов

- `mu_F` - наименьшая неподвижная точка F
- `nu_F` - наибольшая неподвижная точка F
- Для строгих функторов: `mu_F = nu_F` (Haskell с _|_)
- Для тотальных языков: `mu_F != nu_F` (конечные vs бесконечные)

In [57]:
-- ============================================================
-- Демонстрация: гомоморфизм F-алгебр
-- ============================================================

-- h :: A -> B является гомоморфизмом (A, alpha) -> (B, beta)
-- если: beta . fmap h = h . alpha

-- Пример: h :: Int -> String является гомоморфизмом
-- (Int, natToInt) -> (String, natToStr)?
-- Проверим: natToStr . fmap show = show . natToInt

checkHomomorphism :: IO ()
checkHomomorphism = do
  putHeader "Гомоморфизмы F-алгебр"

  putSub "natToInt и natToStr: h = show"
  -- h . alpha = show . natToInt
  -- beta . fmap h = natToStr . fmap show
  let testCases = [ZeroF, SuccF 3, SuccF 0, SuccF 42 :: NatF Int]
  putStrLn "h.alpha vs beta.fmap_h:"
  mapM_ (\f -> do
    let left  = show (natToInt f)
        right = natToStr (fmap show f)
    putStrLn $ "  " ++ show f ++ ": " ++ left ++ " == " ++ right ++ " -> " ++ show (left == right)
    ) testCases

  putSub "Катаморфизм как уникальный гомоморфизм"
  -- Для любой алгебры (A, alpha) существует ровно один гомоморфизм из Fix F
  -- И этот гомоморфизм - cata alpha
  let n = Fix (SuccF (Fix (SuccF (Fix (SuccF (Fix ZeroF))))))
  putStrLn $ "nat 3 -> Int:    " ++ show (cata natToInt n)
  putStrLn $ "nat 3 -> String: " ++ cata natToStr n
  putStrLn $ "nat 3 -> Bool:   " ++ show (cata natIsEven n)

checkHomomorphism

= Гомоморфизмы F-алгебр =

--- natToInt и natToStr: h = show ---
h.alpha vs beta.fmap_h:
  ZeroF: 0 == zero -> False
  SuccF 3: 4 == succ(3) -> False
  SuccF 0: 1 == succ(0) -> False
  SuccF 42: 43 == succ(42) -> False
--- Катаморфизм как уникальный гомоморфизм ---
nat 3 -> Int:    3
nat 3 -> String: succ(succ(succ(zero)))
nat 3 -> Bool:   False

In [58]:
import qualified Data.Map as Map
import Data.Maybe (fromMaybe)
-- ============================================================
-- ФИНАЛЬНАЯ ДЕМОНСТРАЦИЯ: все схемы рекурсии вместе
-- ============================================================
-- Задача: работа с выражениями разными схемами

-- Дерево выражений для демо
sampleExpr :: Expr
sampleExpr = add
  (mul (num 2) (add (var "x") (num 3)))
  (divE (mul (var "y") (num 4)) (num 2))
-- = (2*(x+3)) + ((y*4)/2)

-- Глубина AST через катаморфизм
depthExpr :: Expr -> Int
depthExpr = cata alg
  where
    alg (NumF _)   = 0
    alg (VarF _)   = 0
    alg (AddF l r) = 1 + max l r
    alg (MulF l r) = 1 + max l r
    alg (NegF x)   = 1 + x
    alg (DivF l r) = 1 + max l r

-- Список всех переменных через катаморфизм
varsOf :: Expr -> [String]
varsOf = cata alg
  where
    alg (NumF _)   = []
    alg (VarF v)   = [v]
    alg (AddF l r) = l ++ r
    alg (MulF l r) = l ++ r
    alg (NegF x)   = x
    alg (DivF l r) = l ++ r

-- Подстановка переменных (параморфизм: нужен доступ к имени переменной)
substAlg :: Map.Map String Expr -> ExprF (Expr, Expr) -> Expr
substAlg env (VarF v)       = fromMaybe (var v) (Map.lookup v env)
substAlg _   (NumF n)       = num n
substAlg _   (AddF (_,l) (_,r)) = add l r
substAlg _   (MulF (_,l) (_,r)) = mul l r
substAlg _   (NegF (_,x))   = neg x
substAlg _   (DivF (_,l) (_,r)) = divE l r

subst :: Map.Map String Expr -> Expr -> Expr
subst env = para (substAlg env)

-- Разворачиваем выражение в дерево вычислений (анаморфизм по ExprF)
-- (просто подставляем переменные)
specializeExpr :: Map.Map String Double -> Expr -> Expr
specializeExpr env = para alg
  where
    alg (VarF v)       = fromMaybe (var v) (num <$> Map.lookup v env)
    alg (NumF n)       = num n
    alg (AddF (_,l) (_,r)) = case (unFix l, unFix r) of
      (NumF a, NumF b) -> num (a+b)
      _                -> add l r
    alg (MulF (_,l) (_,r)) = case (unFix l, unFix r) of
      (NumF a, NumF b) -> num (a*b)
      _                -> mul l r
    alg (NegF (_,x))    = case unFix x of
      NumF a -> num (negate a)
      _      -> neg x
    alg (DivF (_,l) (_,r)) = case (unFix l, unFix r) of
      (NumF a, NumF b) | b /= 0 -> num (a/b)
      _                          -> divE l r

finalDemo :: IO ()
finalDemo = do
  putHeader "Финальная демонстрация: все схемы для AST"

  putSub "Исходное выражение"
  putStrLn $ pretty sampleExpr

  putSub "Метаданные (cata)"
  putStrLn $ "Глубина: " ++ show (depthExpr sampleExpr)
  putStrLn $ "Переменные: " ++ show (varsOf sampleExpr)
  let s = countStats sampleExpr
  putStrLn $ "Операций/переменных/констант: " ++
    show (numOps s) ++ "/" ++ show (numVars s) ++ "/" ++ show (numNums s)

  putSub "Подстановка y = x + 1 (para)"
  let subbed = subst (Map.fromList [("y", add (var "x") (num 1))]) sampleExpr
  putStrLn $ pretty subbed

  putSub "Специализация при x=5, y=7 (para + simplify)"
  let env = Map.fromList [("x", 5.0), ("y", 7.0)]
  let specialized = specializeExpr env sampleExpr
  putStrLn $ "После специализации: " ++ pretty specialized

  putSub "Вычисление (cata)"
  putStrLn $ "eval при x=5, y=7: " ++ show (eval env sampleExpr)

finalDemo

Line 54: Use maybe
Found:
fromMaybe (var v) (num <$> Map.lookup v env)
Why not:
maybe (var v) num (Map.lookup v env)

= Финальная демонстрация: все схемы для AST =

--- Исходное выражение ---
((2 * (x + 3)) + ((y * 4) / 2))
--- Метаданные (cata) ---
Глубина: 3
Переменные: ["x","y"]
Операций/переменных/констант: 5/2/4
--- Подстановка y = x + 1 (para) ---
((2 * (x + 3)) + (((x + 1) * 4) / 2))
--- Специализация при x=5, y=7 (para + simplify) ---
После специализации: 30
--- Вычисление (cata) ---
eval при x=5, y=7: Right 30.0

## Бонус: Терминальная коалгебра — Потоки

### Бесконечные структуры через коалгебры

Терминальная F-коалгебра `nu_F` — это наибольшая точка F.
В Haskell это моделируется ленивыми бесконечными структурами.

**Stream** — классический пример:
```haskell
data StreamF a r = StreamF a r  -- всегда бесконечный
type Stream a = Fix (StreamF a) -- ленивый бесконечный поток
```

Анаморфизм для Stream — это аналог `iterate` / `unfold` для бесконечных структур.

Гистоморфизм для Stream — вычисление с историей предыдущих значений (скользящее среднее, рекуррентные последовательности).

In [59]:
-- ============================================================
-- Потоки через коалгебры (конечные приближения)
-- ============================================================

-- Представим Stream как список (конечное приближение)
-- Коалгебра: A -> (a, A)
type StreamCoalg a s = s -> (a, s)

-- Разворачиваем n элементов
unfoldStream :: StreamCoalg a s -> s -> Int -> [a]
unfoldStream _     _    0 = []
unfoldStream coalg seed n =
  let (x, next) = coalg seed
  in x : unfoldStream coalg next (n-1)

-- ============================================================
-- Различные потоки через коалгебры
-- ============================================================

-- Поток натуральных чисел
naturals :: StreamCoalg Int Int
naturals n = (n, n+1)

-- Поток Фибоначчи
fibStream :: StreamCoalg Integer (Integer, Integer)
fibStream (a, b) = (a, (b, a+b))

-- Поток простых чисел (решето Эратосфена через коалгебру)
primesCoalg :: StreamCoalg Int [Int]
primesCoalg (p:rest) = (p, filter (\x -> x `mod` p /= 0) rest)
primesCoalg []       = error "пустое решето"

primes :: Int -> [Int]
primes n = unfoldStream primesCoalg [2..n*20] (min n 30)

-- Поток Коллатца
collatzStream :: Int -> [Int]
collatzStream start = unfoldr step start
  where
    step 1 = Nothing
    step n = let next = if even n then n `div` 2 else 3*n+1
             in Just (n, next)
    unfoldr f b = case f b of
      Nothing     -> []
      Just (a, b') -> a : unfoldr f b'

-- ============================================================
-- Скользящее среднее через гистоморфизм на списке
-- ============================================================
-- movingAvg k xs = список средних по k последних элементов
movingAvg :: Int -> [Double] -> [Double]
movingAvg k xs = go xs
  where
    go [] = []
    go ys
      | length window < k = []
      | otherwise = avg window : go (tail ys)
      where
        window = take k ys
        avg ws = sum ws / fromIntegral (length ws)

streamDemo :: IO ()
streamDemo = do
  putHeader "Потоки через коалгебры (терминальная коалгебра)"

  putSub "Натуральные числа"
  putStrLn $ show (unfoldStream naturals 0 10)

  putSub "Числа Фибоначчи"
  putStrLn $ show (unfoldStream fibStream (0,1) 15)

  putSub "Простые числа"
  putStrLn $ show (primes 20)

  putSub "Последовательность Коллатца от 27"
  let cs = collatzStream 27
  putStrLn $ "Длина: " ++ show (length cs)
  putStrLn $ "Максимум: " ++ show (maximum cs)
  putStrLn $ "Первые 10: " ++ show (take 10 cs)

  putSub "Скользящее среднее (k=3)"
  let prices = [1,3,2,5,4,7,6,8,9,8,10 :: Double]
  putStrLn $ "Цены: " ++ show prices
  putStrLn $ "MA(3): " ++ show (movingAvg 3 prices)

streamDemo

Line 38: Eta reduce
Found:
collatzStream start = unfoldr step start
Why not:
collatzStream = unfoldr stepLine 52: Eta reduce
Found:
movingAvg k xs = go xs
Why not:
movingAvg k = goLine 67: Use print
Found:
putStrLn $ show (unfoldStream naturals 0 10)
Why not:
print (unfoldStream naturals 0 10)Line 70: Use print
Found:
putStrLn $ show (unfoldStream fibStream (0, 1) 15)
Why not:
print (unfoldStream fibStream (0, 1) 15)Line 73: Use print
Found:
putStrLn $ show (primes 20)
Why not:
print (primes 20)

= Потоки через коалгебры (терминальная коалгебра) =

--- Натуральные числа ---
[0,1,2,3,4,5,6,7,8,9]
--- Числа Фибоначчи ---
[0,1,1,2,3,5,8,13,21,34,55,89,144,233,377]
--- Простые числа ---
[2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71]
--- Последовательность Коллатца от 27 ---
Длина: 111
Максимум: 9232
Первые 10: [27,82,41,124,62,31,94,47,142,71]
--- Скользящее среднее (k=3) ---
Цены: [1.0,3.0,2.0,5.0,4.0,7.0,6.0,8.0,9.0,8.0,10.0]
MA(3): [2.0,3.3333333333333335,3.6666666666666665,5.333333333333333,5.666666666666667,7.0,7.666666666666667,8.333333333333334,9.0]

## Взаимная рекурсия и вложенные функторы

### Compose и вложенные алгебры

Мощь F-алгебр в том, что функторы можно **компоновать**:
```haskell
newtype Compose f g a = Compose (f (g a))
```

Это позволяет описывать **взаимно рекурсивные** типы и **вложенные структуры**.

### Rose Tree через вложение

```haskell
data RoseF a r = RoseF a [r]
type Rose a = Fix (RoseF a)
```

Катаморфизм по Rose Tree — это обобщение обхода дерева с переменной arity.

### JSON-подобная структура

JSON — прекрасный пример функтора с несколькими рекурсивными позициями.

In [60]:
import Data.List (intercalate, sort, nub)
-- ============================================================
-- Rose Tree через Fix
-- ============================================================
data RoseF a r = RoseF a [r] deriving (Show, Functor)

type RoseTree a = Fix (RoseF a)

roseLeaf :: a -> RoseTree a
roseLeaf x = Fix (RoseF x [])

roseNode :: a -> [RoseTree a] -> RoseTree a
roseNode x cs = Fix (RoseF x cs)

-- Катаморфизм для Rose Tree
roseSum :: RoseTree Int -> Int
roseSum = cata alg
  where alg (RoseF x rs) = x + sum rs

roseDepth :: RoseTree a -> Int
roseDepth = cata alg
  where alg (RoseF _ []) = 0
        alg (RoseF _ cs) = 1 + maximum cs

roseToList :: RoseTree a -> [a]  -- BFS порядок
roseToList t = bfs [t]
  where
    bfs []     = []
    bfs (x:xs) = let RoseF v cs = unFix x
                 in v : bfs (xs ++ cs)

rosePretty :: Show a => RoseTree a -> String
rosePretty = cata alg
  where
    alg (RoseF x []) = show x
    alg (RoseF x cs) = show x ++ "[" ++ intercalate "," cs ++ "]"

-- Строим Rose Tree через анаморфизм
-- Коалгебра: число -> его делители как дети
divisorTree :: Int -> RoseTree Int
divisorTree = ana coalg
  where
    coalg n = RoseF n (filter (\d -> d > 1 && d < n && n `mod` d == 0) [2..n-1])

-- ============================================================
-- JSON-подобная структура
-- ============================================================
data JsonF r
  = JNull
  | JBool   Bool
  | JNum    Double
  | JStr    String
  | JArr    [r]
  | JObj    [(String, r)]
  deriving (Functor)

type Json = Fix JsonF

-- Умные конструкторы
jNull :: Json
jNull = Fix JNull
jBool :: Bool -> Json
jBool = Fix . JBool
jNum :: Double -> Json
jNum = Fix . JNum
jStr :: String -> Json
jStr = Fix . JStr
jArr :: [Json] -> Json
jArr = Fix . JArr
jObj :: [(String, Json)] -> Json
jObj = Fix . JObj

-- Красивая печать JSON (катаморфизм)
prettyJson :: Json -> String
prettyJson = cata alg
  where
    alg JNull        = "null"
    alg (JBool b)    = if b then "true" else "false"
    alg (JNum n)     = if n == fromIntegral (round n :: Int) then show (round n :: Int) else show n
    alg (JStr s)     = "\"" ++ s ++ "\""
    alg (JArr xs)    = "[" ++ intercalate ", " xs ++ "]"
    alg (JObj kvs)   = "{" ++ intercalate ", " (map (\(k,v) -> "\""++k++"\": "++v) kvs) ++ "}"

-- Подсчёт элементов
countJson :: Json -> Int
countJson = cata alg
  where
    alg JNull        = 1
    alg (JBool _)    = 1
    alg (JNum _)     = 1
    alg (JStr _)     = 1
    alg (JArr xs)    = 1 + sum xs
    alg (JObj kvs)   = 1 + sum (map snd kvs)

roseAndJsonDemo :: IO ()
roseAndJsonDemo = do
  putHeader "Rose Tree и JSON через F-алгебры"

  putSub "Rose Tree"
  let t = roseNode 1 [roseNode 2 [roseLeaf 4, roseLeaf 5], roseNode 3 [roseLeaf 6]]
  putStrLn $ "Структура: " ++ rosePretty t
  putStrLn $ "BFS: " ++ show (roseToList t)
  putStrLn $ "Сумма: " ++ show (roseSum t)
  putStrLn $ "Глубина: " ++ show (roseDepth t)

  putSub "Дерево делителей 12"
  let dt = divisorTree 12
  putStrLn $ rosePretty dt

  putSub "JSON через Fix"
  let person = jObj
        [ ("name", jStr "Alice")
        , ("age", jNum 30)
        , ("hobbies", jArr [jStr "Haskell", jStr "категорий теория"])
        , ("active", jBool True)
        , ("score", jNull)
        ]
  putStrLn $ prettyJson person
  putStrLn $ "Узлов в JSON: " ++ show (countJson person)

roseAndJsonDemo

= Rose Tree и JSON через F-алгебры =

--- Rose Tree ---
Структура: 1[2[4,5],3[6]]
BFS: [1,2,3,4,5,6]
Сумма: 21
Глубина: 2
--- Дерево делителей 12 ---
12[2,3,4[2],6[2,3]]
--- JSON через Fix ---
{"name": "Alice", "age": 30, "hobbies": ["Haskell", "категорий теория"], "active": true, "score": null}
Узлов в JSON: 8

---

**← Предыдущий:** [Трансформеры комонад](ComonadTransformers.ipynb)  |  **Следующий →** [Профункторы](Profunctors.ipynb)
